In [1]:
import langchain
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic import hub
import os
langchain.__version__


'1.5.0'

In [2]:
with open("data/secrets/key.secret") as f:
    os.environ["OPENAI_API_KEY"] = f.read()

In [5]:
model = init_chat_model("gpt-5.6-luna", model_provider="openai")
model

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.8.0', 'langchain': '1.5.0', 'langchain-openai': '1.7.4'}}, profile={'name': 'GPT-5.6 Luna', 'release_date': '2026-07-09', 'last_updated': '2026-07-09', 'open_weights': False, 'max_input_tokens': 1050000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['none', 'low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'medium', 'mid_conversation_system_messages': True, 'mid_conversation_tools': False}, client=<openai.resources.chat.completions.completions.Completions o

In [6]:
# Ne fonctionne pas car non chainé
model.invoke([{"role": "system", "content": "Traduis ce texte du Français à l'Italien"}])
response=model.invoke([{"role": "user", "content": "Salut, je m'appelle Cyril"}])
response.content

'Salut Cyril ! Enchanté de faire ta connaissance 😊 Comment puis-je t’aider aujourd’hui ?'

In [7]:
# Chainage
system_message={"role": "system", "content": "Traduis ce texte du Français à l'Italien"}
user_message={"role": "user", "content": "Salut, je m'appelle Cyril"}
response=model.invoke([system_message,user_message])
response.content

'Ciao, mi chiamo Cyril.'

In [8]:
# Autre écriture avec une logique OO Langchain
messages = [
    SystemMessage(content="Traduis ce texte du Français à l'Italien"),
    HumanMessage(content="Salut, je m'appelle Cyril"),
]
response=model.invoke(messages)
response.content

'Ciao, mi chiamo Cyril.'

In [9]:
# Template
system_template = "Traduis ce texte du Francais vers {language}"
prompt_template = ChatPromptTemplate.from_messages(
    [("system", system_template),  ("user", "{text}")]
)
prompt_template

ChatPromptTemplate(input_variables=['language', 'text'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['language'], input_types={}, partial_variables={}, template='Traduis ce texte du Francais vers {language}'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['text'], input_types={}, partial_variables={}, template='{text}'), additional_kwargs={})])

In [10]:
prompt = prompt_template.invoke({"language": "Italian", "text": "Salut, je m'appelle Cyril"})
prompt

ChatPromptValue(messages=[SystemMessage(content='Traduis ce texte du Francais vers Italian', additional_kwargs={}, response_metadata={}), HumanMessage(content="Salut, je m'appelle Cyril", additional_kwargs={}, response_metadata={})])

In [11]:
response = model.invoke(prompt)
response.content

'Ciao, mi chiamo Cyril.'

HUB RLM
Templates préfabriqués sur le cloud

rlm.rag-prompt : https://smith.langchain.com/hub/rlm/rag-prompt?_gl=1*11dcy42*_gcl_au*MTAxOTgzMzI4My4xNzYwMzQ5ODk5*_ga*MTI3MDkwMDAzMS4xNzYwMzQ5ODk5*_ga_47WX3HKKY2*czE3NjA2MzYyOTEkbzgkZzEkdDE3NjA2MzczMzkkajI0JGwwJGgw

In [17]:
# prompt_template = hub.pull("rlm/rag-prompt") # Deprecated security issue
# prompt_template
from langsmith import Client
client = Client()
prompt_template = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)


In [18]:
prompt = prompt_template.invoke({"context": "Le Hub RLM est une base de données de ChatPromptTemplate déjà écrits et disponibles sur https://smith.langchain.com/hub/rlm/, répondre en 5 mots", "question": "Qu'est que le HUB RLM ?"})
response = model.invoke(prompt)
response.content

'Base de ChatPromptTemplate déjà écrits'